In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


rng = np.random.default_rng(42)

n_samples = 2000
n_features = 1000
n_true_factors = 200  # the real underlying dimensionality


# 200 true factors, combined randomly to create 1000 observed
# (noisy, redundant) features

true_factors = rng.normal(
    0,
    1,
    size=(n_samples, n_true_factors)
)

mixing_weights = rng.normal(
    0,
    1,
    size=(n_true_factors, n_features)
)

X = (
    true_factors @ mixing_weights
    + rng.normal(
        0,0.5,size=(n_samples, n_features)
    )
)


# The target depends on those SAME true factors (plus noise)
# -- this is why PCA won't throw away the signal,
# only the redundant noise

true_coefficients = rng.normal(
    0,1,size=n_true_factors)

y = (
    true_factors @ true_coefficients
    + rng.normal(
        0,5,size=n_samples
    )
)
print(
    f"X shape: {X.shape} "
    f"(1000 columns, but only {n_true_factors} are truly independent)"
)

## Step 2 — Train/Test Split + Baseline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.25,random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


baseline_model = LinearRegression().fit(
    X_train_scaled,
    y_train
)


r2_baseline = r2_score(
    y_test,
    baseline_model.predict(X_test_scaled)
)


print(
    f"Linear Regression on all {X.shape[1]} raw features "
    f"-> test R² = {r2_baseline:.3f}"
)

## Step 3 — Apply PCA and look at explained variance

In [ ]:
pca_full = PCA(random_state=42)

pca_full.fit(X_train_scaled)

cumulative_variance = np.cumsum(
    pca_full.explained_variance_ratio_
)

for n_comp in [50, 100, 150, 200, 250, 300]:
    print(
        f"First {n_comp:4d} components explain "
        f"{cumulative_variance[n_comp-1]:.1%} of the total variance"
    )

Step 4 Scree Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(range(1, 301), pca_full.explained_variance_ratio_[:300], color="#028090")
axes[0].axvline(200, color="#F96167", linestyle="--", label="200 components")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title("Scree plot (first 300 components)")
axes[0].legend()

axes[1].plot(range(1, 301), cumulative_variance[:300], color="#028090")
axes[1].axvline(200, color="#F96167", linestyle="--", label="200 components")
axes[1].axhline(0.99, color="gray", linestyle=":", label="99% variance")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative explained variance")
axes[1].set_title("Cumulative explained variance")
axes[1].legend()

plt.tight_layout()

Step 5 

In [ ]:
results = {}
for n_comp in [50, 100, 150, 200, 250, 300]:
    pca = PCA(n_components=n_comp, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    model = LinearRegression().fit(X_train_pca, y_train)
    r2 = r2_score(y_test, model.predict(X_test_pca))
    results[n_comp] = r2
    
    print(f"PCA with {n_comp:4d} components -> test R^2 = {r2:.3f}")

print(f"\nFor comparison, all {X.shape[1]} raw features -> test R^2 = {r2_baseline:.3f}")